## Config 1.0 Importing libraries

In [4]:
import torch
import pandas as pd
import numpy as np
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, VGAE, SAGEConv
from torch_geometric.utils import train_test_split_edges
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm
import os
from collections import defaultdict
import gzip
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.decomposition import IncrementalPCA
from scipy import sparse
import re
from pyfaidx import Fasta # Assuming pyfaidx is installed
from functools import reduce

c:\Users\Urveesh\Desktop\GNN\PythonVenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Node Creation

Start with unifying the data


In [ ]:
## BELLOW CODE IS TO BE RUN IN R SERIAL WISE
# # --- R Code to Convert Specific .Rdata File Assay to CSV ---

# # --- 1. Ensure Required Packages Are Loaded ---
# # Make sure you have installed these packages previously using BiocManager:
# # install.packages("BiocManager", repos="https://cloud.r-project.org")
# # library(BiocManager)
# # BiocManager::install(c("SummarizedExperiment", "Biobase"), ask=FALSE)

# library(SummarizedExperiment)
# # library(Biobase) # Load Biobase just in case SummarizedExperiment object needs it

# # --- 2. Configuration: File Paths ---

# # Set the full path to your input .Rdata file
# # Use forward slashes (/) or double backslhes (\\) for paths in R on Windows
# input_rdata_file <- "C:/Users/Urveesh/Desktop/GNN/data/brainseq_neuron2019/rse_gene_unfiltered.Rdata"

# # Set the full path for your desired output CSV file in the same directory
# output_csv_file <- "C:/Users/Urveesh/Desktop/GNN/data/brainseq_neuron2019/tpm_gene_unfiltered.csv"

# # --- 3. Load the Rdata file ---
# # This loads object(s) into R's memory. Based on ls() output, expects 'rse_gene'.
# print(paste0("Attempting to load data from: ", input_rdata_file))

# loading_successful <- tryCatch({
#   load(input_rdata_file, envir = .GlobalEnv)
#   TRUE
# }, error = function(e) {
#   print(paste0("ERROR: Failed to load Rdata file. Message: ", e$message))
#   print("Ensure sufficient RAM.")
#   FALSE
# })

# if (!loading_successful) {
#   stop("Script stopped because Rdata file could not be loaded.")
# }
# print("Rdata file loaded successfully.")

# # --- 4. Access the Loaded Object and Extract the Correct Assay ---
# # Based on your ls() output, the main object is 'rse_gene'.
# # Check the actual assay names using assayNames(rse_gene) in R console.

# if (!exists("rse_gene")) {
#   print("ERROR: Object 'rse_gene' not found after loading. Check file path and contents.")
#   stop("Required object not found.")
# }

# if (!is(rse_gene, "SummarizedExperiment")) {
#   print("Warning: 'rse_gene' is not a SummarizedExperiment. Assay extraction might fail.")
# } else {
#   print("Object 'rse_gene' is a SummarizedExperiment.")
#   # Print available assay names to help debugging if needed
#   print("Available assay names:")
#   print(assayNames(rse_gene)) # <-- This will show you the correct names
# }

# print("Extracting assay...")
# # *** REPLACE "Your_Actual_Assay_Name_Here" with the exact name from assayNames(rse_gene) output ***
# actual_assay_name <- "rpkm" # <--- !!! EDIT THIS LINE !!!

# if (!(actual_assay_name %in% assayNames(rse_gene))) {
#   print(paste0("ERROR: Assay '", actual_assay_name, "' not found in object 'rse_gene'."))
#   print("Available assay names are:")
#   print(assayNames(rse_gene))
#   stop("Assay not found.")
# }

# tpm_data_frame <- as.data.frame(assay(rse_gene, actual_assay_name)) # <--- Uses the variable with the correct name
# print(paste0("Assay '", actual_assay_name, "' extracted successfully."))

# # --- 5. Write the data frame to CSV ---
# print(paste0("Writing data frame to CSV: ", output_csv_file))

# write.csv(tpm_data_frame, file = output_csv_file, row.names = TRUE, quote = FALSE)

# print("✅ CSV file generated successfully.")

# # --- Optional: Clean up ---
# # rm(list = c("rse_gene", "tpm_data_frame"))

In [ ]:
data_root = "data/brainseq_neuron2019"
csv_files = [
    f"{data_root}/tpm_gene_unfiltered.csv",
    f"{data_root}/tpm_tx_unfiltered.csv",
    f"{data_root}/rse_exon_unfiltered.csv",
    f"{data_root}/rse_jxn_unfiltered.csv",
]

expr_ids = set()
for path in csv_files:
    df = pd.read_csv(path, nrows=0, index_col=0)  # just read header
    idx = pd.read_csv(path, index_col=0).index  # full index
    # strip versions, uppercase
    clean = idx.to_series().str.upper().str.split(".").str[0]
    expr_ids |= set(clean)

print(f"Total unique expression IDs: {len(expr_ids):,}")


In [ ]:
#GENERATING ensembl_to_rnacentral FEATURES#
# 1. Prepare to stream id_mapping.tsv
colnames = ["rnacentral_id","database","external_id","other_acc","taxid","rna_type"]
chunks = pd.read_csv(
    "data/rnacentral/id_mapping.tsv",
    sep="\t",
    names=colnames,
    header=None,
    chunksize=200_000,
    dtype=str,
)

# 2. Filter for Ensembl IDs (external_id starting with ENSG)
maps = []
for chunk in chunks:
    ens = chunk[chunk["external_id"].str.startswith("ENSG")]
    if not ens.empty:
        maps.append(ens[["external_id","rnacentral_id"]])

# 3. Concatenate and dedupe
ens_map = pd.concat(maps, ignore_index=True)
ens_map = ens_map.drop_duplicates(subset=["external_id"])

# 4. Save to CSV
out_path = "data/brainseq_neuron2019/ensembl_to_rnacentral.csv"
ens_map.to_csv(out_path, index=False)
print(f"✅ Built Ensembl → RNAcentral map with {len(ens_map):,} entries")


In [ ]:
#GENERATING node_mndr_features FEATURES#
# Load MNDR and clean
xls = pd.ExcelFile("data/mndr/RNADiseasev4.0_RNA-disease_experiment_all.xlsx")
df_mndr = xls.parse(sheet_name=xls.sheet_names[0])
df_mndr["RNA Symbol"] = (
    df_mndr["RNA Symbol"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(r"\.\d+$", "", regex=True)
)
mndr_symbols = set(df_mndr["RNA Symbol"])
import pandas as pd

# Column names (no header line in file)
colnames = [
    "rnacentral_id",    # col 0
    "database",         # col 1
    "external_id",      # col 2
    "other_acc",
    "taxid",
    "rna_type"
]

# Dictionary to collect matches
mapping = {}

# Stream‐read in chunks
for chunk in pd.read_csv(
    "data/rnacentral/id_mapping.tsv",
    sep="\t",
    names=colnames,
    usecols=["rnacentral_id", "external_id"],
    header=None,
    chunksize=100_000,
    iterator=True,
    dtype=str,
):
    # Uppercase and strip the external IDs in this chunk
    chunk["external_id"] = chunk["external_id"].str.strip().str.upper()
    # Filter to only your MNDR symbols
    sub = chunk[chunk["external_id"].isin(mndr_symbols)]
    # Add into mapping dict
    for ext, rc in zip(sub["external_id"], sub["rnacentral_id"]):
        mapping[ext] = rc

print(f"✅ Found {len(mapping)} RNAcentral mappings out of {len(mndr_symbols)} MNDR symbols")
from sklearn.preprocessing import MinMaxScaler

# Turn mapping into a DataFrame
df_map = pd.DataFrame(mapping.items(), columns=["RNA Symbol", "rnacentral_id"])

# Merge with your MNDR DataFrame
df_nodes = pd.merge(
    df_mndr,
    df_map,
    on="RNA Symbol",
    how="inner",
)
print(f"✅ Merged rows: {len(df_nodes)}")

# Normalize the score
scaler = MinMaxScaler()
df_nodes["mndr_score"] = scaler.fit_transform(df_nodes[["score"]])

# Final node feature table
node_features = df_nodes[["rnacentral_id", "mndr_score"]].drop_duplicates()
out_csv = "data/mndr/node_mndr_features.csv"
node_features.to_csv(out_csv, index=False)
print(f"✅ node_mndr_features.csv written with {len(node_features)} nodes")


In [ ]:
mapping_files = [
    "data/rnacentral/id_mapping/database_mappings/ensembl.tsv.gz",
    "data/rnacentral/id_mapping/database_mappings/ensembl_gencode.tsv.gz",
]

maps = []
for fn in mapping_files:
    df = pd.read_csv(
        fn,
        sep="\t",
        names=["rnacentral_id", "external_id"],
        compression="gzip",
        dtype=str,
    )
    df["external_id"] = df["external_id"].str.upper().str.split(".").str[0]
    maps.append(df[["external_id","rnacentral_id"]])

expr_map_df = (
    pd.concat(maps, ignore_index=True)
      .drop_duplicates(subset=["external_id"])
)
print(f"✅ Loaded {len(expr_map_df):,} Ensembl → RNAcentral mappings")


In [ ]:
#GENERATING node_features FEATURES#


data_root = "data/brainseq_neuron2019"
csv_files = {
    "gene": f"{data_root}/tpm_gene_unfiltered.csv",
    "tx":   f"{data_root}/tpm_tx_unfiltered.csv",
    "exon": f"{data_root}/rse_exon_unfiltered.csv",
    "jxn":  f"{data_root}/rse_jxn_unfiltered.csv",
}

def process_expr(path, prefix):
    print(f"🔹 {prefix}: loading…")
    df = pd.read_csv(path, index_col=0)
    df = np.log2(df + 1)
    df.index = (
        df.index.to_series().str.upper().str.split(".").str[0]
    )
    df = df.reset_index().rename(columns={"index":"external_id"})
    merged = df.merge(expr_map_df, on="external_id", how="inner")
    node = (
        merged
        .drop(columns=["external_id"])
        .groupby("rnacentral_id")
        .mean()
        .reset_index()
        .rename(columns=lambda c: f"{prefix}_{c}" if c!="rnacentral_id" else c)
    )
    print(f"✅ {prefix} mapped: {len(node):,} nodes")
    return node

parts = [process_expr(p, k) for k,p in csv_files.items()]

print("\n🔗 Merging modalities…")
node_expr = reduce(lambda a,b: a.merge(b, on="rnacentral_id", how="outer"),
                   parts).fillna(0)
node_expr.to_csv(f"{data_root}/node_brainseq_features.csv", index=False)
print("✅ node_brainseq_features.csv shape:", node_expr.shape)

# Integrate MNDR scores
mndr = pd.read_csv("data/mndr/node_mndr_features.csv")
all_nodes = mndr.merge(node_expr, on="rnacentral_id", how="outer").fillna(0)
all_nodes.to_csv("data/node_features.csv", index=False)
print("🎯 node_features.csv shape:", all_nodes.shape)


End code of node creation

In [ ]:
# --- 1. Load HGNC complete set (symbol ↔ hgnc_id) ---

# Use the new GCS URL
hgnc_url = "https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt"
print(f"Loading HGNC complete set from: {hgnc_url}")

# Only pull the columns we need
hgnc = pd.read_csv(
    hgnc_url,
    sep="\t",
    usecols=["hgnc_id","symbol"],
    dtype=str
)
# Strip the 'HGNC:' prefix
hgnc["hgnc_id"] = hgnc["hgnc_id"].str.replace(r"^HGNC:","",regex=True)

print(f"✅ Loaded HGNC complete set with {len(hgnc):,} entries.")
print(hgnc.head())

# --- 2. Load only columns 0 and 2 from RNAcentral HGNC map ---
hgnc_map_file = "data/rnacentral/id_mapping/database_mappings/hgnc.tsv"
hgnc_map = pd.read_csv(
    hgnc_map_file,
    sep="\t",
    header=None,
    usecols=[0,2],                   # Column 0: URS ID, Column 2: HGNC accession
    names=["rnacentral_id","hgnc_id"],
    dtype=str
)
# Strip 'HGNC:' prefix from `hgnc_id` just in case
hgnc_map["hgnc_id"] = hgnc_map["hgnc_id"].str.replace(r"^HGNC:","",regex=True)

# --- 3. Merge to get symbol → rnacentral_id ---
symbol_to_urs = (
    pd.merge(
        hgnc,
        hgnc_map,
        on="hgnc_id",
        how="inner"
    )
    [["symbol","rnacentral_id"]]
    .drop_duplicates(subset=["symbol"])
)
print(f"✅ Built symbol→URS map: {len(symbol_to_urs):,} entries")
# e.g. expect ≫ 10,000 mapped symbols


In [ ]:
# ─── 1. Load & preprocess gene‑level TPMs ──────────────────────────────
tpm_path = "data/brainseq_neuron2019/tpm_gene_unfiltered.csv"
gene_df = pd.read_csv(tpm_path, index_col=0)
gene_df = np.log2(gene_df + 1)

# Strip version from ENSG IDs and reset index to a column named "gene_id"
gene_df.index = (
    gene_df.index
    .to_series()
    .str.upper()           # uppercase for consistency
    .str.split(".")        # split off version suffix
    .str[0]
)
gene_df = gene_df.reset_index().rename(columns={"index": "gene_id"})
print(f"Loaded gene TPMs: {gene_df.shape[0]:,} genes × {gene_df.shape[1]-1:,} samples")

# ─── 2. Parse GENCODE GTF for gene_id → gene_name ──────────────────────
gtf_file = "data/gencode.v43.annotation.gtf.gz"
gene_rows = []
with gzip.open(gtf_file, "rt") as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        chrom, src, feature, start, end, score, strand, phase, attrs = line.split("\t", 8)
        if feature != "gene":
            continue
        # Extract attributes via regex
        m = dict(re.findall(r'(\S+) "([^"]+)"', attrs))
        gene_id   = m["gene_id"].split(".")[0]
        gene_name = m["gene_name"].upper()
        gene_rows.append((gene_id, gene_name))
gtf_genes = (
    pd.DataFrame(gene_rows, columns=["gene_id","symbol"])
      .drop_duplicates(subset=["gene_id"])
)
print(f"Parsed GTF genes: {gtf_genes.shape[0]:,}")

# ─── 3. Build symbol → URS map via HGNC ────────────────────────────────
# (Assumes you've already run the corrected HGNC code and have `symbol_to_urs` DF)
symbol_to_urs = pd.read_csv("data/rnacentral/symbol_to_urs.csv")  
# symbol_to_urs has columns ['symbol','rnacentral_id'], ~8,744 rows

# ─── 4. Merge gene_df → gtf_genes → symbol_to_urs ────────────────────
# 4.1 gene_df + gtf_genes on 'gene_id'
tmp = pd.merge(gene_df, gtf_genes, on="gene_id", how="inner")
print(f"After GTF merge: {tmp.shape[0]:,} genes")

# 4.2 tmp + symbol_to_urs on 'symbol'
merged = pd.merge(tmp, symbol_to_urs, on="symbol", how="inner")
print(f"After HGNC mapping: {merged['rnacentral_id'].nunique():,} unique nodes")

# 4.3 Collapse by rnacentral_id to get per‑node features
node_gene = (
    merged
    .drop(columns=["gene_id","symbol"])
    .groupby("rnacentral_id")
    .mean()
    .reset_index()
)
print(f"✅ Mapped gene‑level features: {node_gene.shape[0]:,} nodes × {node_gene.shape[1]-1:,} features")

# 4.4 Save
node_gene.to_csv("data/brainseq_neuron2019/node_gene_features.csv", index=False)


In [ ]:
#GENERATING node_augmented_features FEATURES#

# Cell X+2: Compute GTF‐derived gene length and merge into node features
# Paths – adjust as needed
GTF_FILE         = "data/gencode/gencode.v43.annotation.gtf" #download
ENSEMBL_MAP      = "data/brainseq_neuron2019/ensembl_to_rnacentral.csv" #mapped back but confirmation is needed
OUT_FILE    = "data/node_augmented_features.csv"
# Cell: Rebuild node features from edge‐list + all sources + gene_length

# Paths
EDGE_FILE       = "data/rnacentral_edges.csv"
FEATURE_FILES   = {
    "gene":  "data/brainseq_neuron2019/node_gene_features.csv",#mapped back
    "brain": "data/brainseq_neuron2019/node_brainseq_features.csv",#mapped but highly unsure(So this may help you out)
    "mdnr":  "data/mndr/node_mndr_features.csv",#mapped back
    "other": "data/node_features.csv",#mapped back
}


# 1) Build list of all URS nodes in the pruned graph
edges     = pd.read_csv(EDGE_FILE)
node_ids  = sorted(set(edges.source_URS) | set(edges.target_URS))
nodes_df  = pd.DataFrame({"rnacentral_id": node_ids})
print(f"⦿ Preparing features for {len(nodes_df)} graph nodes")

# 2) For each feature file: merge & mark presence
for tag, path in FEATURE_FILES.items():
    df_tag = pd.read_csv(path)
    # remember which URS appear in this file
    present_set = set(df_tag["rnacentral_id"])
    # merge in all columns from df_tag (except duplicates of rnacentral_id)
    nodes_df = nodes_df.merge(
        df_tag,
        on="rnacentral_id",
        how="left",
        suffixes=("", f"_{tag}")
    )
    # mark presence flag
    nodes_df[f"has_{tag}"] = nodes_df["rnacentral_id"].isin(present_set)

# 3) Fill missing numeric cells with 0, and leave has_* booleans intact
for col in nodes_df.columns:
    if col.startswith("has_") or col == "rnacentral_id":
        continue
    # numeric or object columns → fillna(0)
    nodes_df[col] = nodes_df[col].fillna(0)

# 4) Compute gene_length from GTF
gene_exons = defaultdict(int)
with open(GTF_FILE) as fh:
    for line in fh:
        if not line.startswith("#"):
            chrom, src, ftype, start, end, score, strand, phase, attrs = line.strip().split("\t", 8)
            if ftype == "exon":
                d = {kv.split(" ")[0]: kv.split(" ")[1].strip('"')
                     for kv in attrs.split(";") if kv.strip()}
                ensg = d.get("gene_id", "").split(".")[0]
                if ensg:
                    gene_exons[ensg] += (int(end) - int(start) + 1)

emap = pd.read_csv(ENSEMBL_MAP, dtype=str)
emap["external_id"] = emap["external_id"].str.split(".").str[0]

# map URS → mean exon length
urs2lens = defaultdict(list)
for _, row in emap.iterrows():
    urn = row["rnacentral_id"]
    ensg = row["external_id"]
    length = gene_exons.get(ensg, 0)
    urs2lens[urn].append(length)

urs2len = {u: sum(lens)/len(lens) for u, lens in urs2lens.items()}

nodes_df["gene_length"] = nodes_df["rnacentral_id"].map(urs2len).fillna(0)

# 5) Save augmented features
nodes_df.to_csv(OUT_FILE, index=False)
print(f"✔️ Saved augmented node features → {OUT_FILE}")
print(f"Columns now include: {list(nodes_df.columns[:5])} … [+ {len(nodes_df.columns)-5} more]")
print(f"Sample head:\n{nodes_df.head()}")

In [ ]:
#GENERATING node_with_hashkmers FEATURES#
# Cell: Memory‐safe k-mer Hashing + Incremental PCA (fixed)
# 1) Load your graph node URS list and sequences (streaming)
edges = pd.read_csv("data/rnacentral_edges.csv")
nodes = set(edges.source_URS) | set(edges.target_URS)

def load_fasta_iter(fasta_path, target_ids):
    with open(fasta_path) as fh:
        seq_id, chunks = None, []
        for line in fh:
            line = line.rstrip()
            if line.startswith(">"):
                if seq_id in target_ids:
                    yield seq_id, "".join(chunks)
                parts = line[1:].split()
                seq_id, chunks = parts[0], []
            else:
                if seq_id in target_ids:
                    chunks.append(line)
        if seq_id in target_ids:
            yield seq_id, "".join(chunks)

seq_iter = load_fasta_iter("data/rnacentral/rnacentral_active.fasta", nodes)

# 2) Build a hashing vectorizer for 6-mers
def seq_to_kmers(seq, k=6):
    return (seq[i:i+k] for i in range(len(seq) - k + 1))

hv = HashingVectorizer(
    analyzer='word',
    n_features=2**18,       # ~262k dims—adjust if needed
    norm='l2',
    alternate_sign=False
)

# 3) Stream each sequence, transform to sparse row, collect into CSR
rows, ids = [], []
for urs, seq in seq_iter:
    kmers = " ".join(seq_to_kmers(seq, k=6))
    row = hv.transform([kmers])    # returns 1×n_features sparse
    rows.append(row)
    ids.append(urs)

X_hashed = sparse.vstack(rows)     # CSR matrix of shape (n_nodes, n_features)
print(f"✔️ Hashed TF rows: {X_hashed.shape}")

# 4) Incremental PCA to reduce to 128 dims
n_components   = 128
pca_batch_size = 256  # must be >= n_components

ipca = IncrementalPCA(n_components=n_components, batch_size=pca_batch_size)

# 4a) partial_fit on all chunks
for start in range(0, X_hashed.shape[0], pca_batch_size):
    end   = min(start + pca_batch_size, X_hashed.shape[0])
    batch = X_hashed[start:end].toarray()
    ipca.partial_fit(batch)

# 4b) transform each chunk
reduced_rows = []
for start in range(0, X_hashed.shape[0], pca_batch_size):
    end   = min(start + pca_batch_size, X_hashed.shape[0])
    batch = X_hashed[start:end].toarray()
    reduced = ipca.transform(batch)
    reduced_rows.append(reduced)

X_reduced = np.vstack(reduced_rows)
print(f"✔️ Reduced to shape: {X_reduced.shape}")

# 5) Merge into your node features
feat      = pd.read_csv("data/node_augmented_features.csv", index_col="rnacentral_id")
embed_df  = pd.DataFrame(X_reduced, index=ids,
                         columns=[f"km_reduced_{i}" for i in range(n_components)])
combined  = feat.join(embed_df, how="left").fillna(0)

# 6) Save
combined.to_csv("data/node_with_hashkmers.csv")
print("✔️ Saved node_with_hashkmers.csv")


In [ ]:
#GENERATING node_with_rnabert FEATURES#
# Cell: RNABERT Embedding with PyTorch check & diagnostics

print("⦿ Checking PyTorch import…")
embeddings = {}
processed_count = 0 # Initialize processed_count outside try block
try:
    # Check for GPU - will use CPU if no CUDA device is found
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"✔️ PyTorch version: {torch.__version__}")
    print(f"ℹ️ Using device: {device}") # This will show 'cpu'

    # Set environment variable to potentially mitigate symlink warning (optional)
    os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

    print("⦿ Importing Hugging Face RNABERT (from multimolecule)…")
    try:
        from multimolecule import RnaTokenizer, RnaBertModel

        # Use the specific classes directly
        # Consider adding cache_dir if you want to control download location
        # cache_dir = "./hf_cache"
        # tokenizer = RnaTokenizer.from_pretrained("multimolecule/rnabert", cache_dir=cache_dir)
        # model = RnaBertModel.from_pretrained("multimolecule/rnabert", cache_dir=cache_dir)

        tokenizer = RnaTokenizer.from_pretrained("multimolecule/rnabert")
        model = RnaBertModel.from_pretrained("multimolecule/rnabert")

        # Move model to device (will move to CPU if device is 'cpu')
        model.to(device)

        # You can optionally print the model's max position embeddings to confirm
        print(f"ℹ️ Model's max_position_embeddings: {model.config.max_position_embeddings}")
        # Set the model_max_length based on the model's config
        model_max_length = model.config.max_position_embeddings # Should be 440

        model.eval()

        # Load only your graph node sequences (reuse your indexed loader)
        graph_ids = [] # Initialize graph_ids list
        try:
            print("⦿ Loading graph edges: data/rnacentral_edges.csv...") # Added print for edges
            edges = pd.read_csv("data/rnacentral_edges.csv")
            graph_ids = sorted(list(set(edges.source_URS) | set(edges.target_URS))) # Use sorted list for consistent order
            print(f"✔️ Found {len(graph_ids)} unique graph nodes from edges.")
        except FileNotFoundError:
            print("❌ Error: data/rnacentral_edges.csv not found. Cannot determine graph nodes.")
            # graph_ids remains empty

        fasta = None # Initialize fasta to None
        if graph_ids: # Only proceed to load FASTA if graph_ids were successfully read
             try:
                 # --- This is the potentially slow step that cannot use tqdm directly ---
                 print(f"⦿ Loading and indexing FASTA file: data/rnacentral/rnacentral_active.fasta (This may take some time)...")
                 fasta = Fasta("data/rnacentral/rnacentral_active.fasta")
                 # --- This print confirms the FASTA loading/indexing is complete ---
                 print(f"✔️ Loaded FASTA file. Contains {len(fasta)} sequences.")

             except FileNotFoundError:
                  print("❌ Error: data/rnacentral/rnacentral_active.fasta not found. Cannot retrieve sequences.")
                  graph_ids = [] # Set to empty to skip embedding generation
             except Exception as e:
                 print(f"❌ Error loading FASTA file: {e}")
                 graph_ids = []


        if graph_ids and fasta: # Only proceed if both graph_ids and fasta are available
            print(f"⦿ Generating RNABERT embeddings for {len(graph_ids)} graph nodes…")

            # --- Batch Processing Setup ---
            batch_size = 32 # Adjust based on your CPU/RAM, try 16 or 64 if needed
            sequences_to_process = []
            urs_order = [] # To keep track of URS in the batch

            # Use tqdm for a nice progress bar over the sequences
            # This bar shows progress for the embedding generation loop
            for i, urs in tqdm(enumerate(graph_ids), total=len(graph_ids), desc="Generating Embeddings"):
                if urs not in fasta:
                    # print(f"⚠️ Sequence for {urs} not found in FASTA. Skipping.") # Optional: for debugging
                    continue

                seq = str(fasta[urs][:].seq)

                sequences_to_process.append(seq)
                urs_order.append(urs)

                # Process batch when it's full or at the end of the loop
                if len(sequences_to_process) == batch_size or (i == len(graph_ids) - 1 and sequences_to_process):

                    try:
                        # --- Batch Tokenization ---
                        inputs = tokenizer(sequences_to_process,
                                           return_tensors="pt",
                                           padding="max_length", # Pad to max_length
                                           truncation=True,      # Truncate if longer
                                           max_length=model_max_length) # Use 440

                        # --- Move inputs to device (will be CPU) ---
                        inputs = {key: val.to(device) for key, val in inputs.items()}

                        with torch.no_grad():
                            out = model(**inputs)

                        # --- Process Batch Outputs ---
                        # Use mean pooling over sequence length dim
                        # results will be on CPU if device was 'cpu'
                        batch_embeddings = out.last_hidden_state.mean(dim=1).cpu().numpy()

                        # Assign embeddings to the correct URS IDs from the batch
                        for j, current_urs in enumerate(urs_order):
                             embeddings[current_urs] = batch_embeddings[j]
                             # processed_count += 1 # No need to increment here, tqdm tracks progress

                        # Clear lists for the next batch
                        sequences_to_process = []
                        urs_order = []

                    except Exception as e:
                         # Using \n before the print to avoid interfering with tqdm line
                         print(f"\n❌ Error processing a batch (starting with {urs_order[0]}): {e}")
                         # Optionally clear the batch and continue to the next one
                         sequences_to_process = []
                         urs_order = []

            processed_count = len(embeddings) # Update processed_count after the loop
            print(f"✔️ Finished generating embeddings. Generated {len(embeddings)} embeddings.")

    except ImportError:
        print("❌ Failed to import RnaTokenizer/RnaBertModel from 'multimolecule'.")
        print("   Make sure the 'multimolecule' package is installed (`pip install multimolecule`).")
    except Exception as e:
        print(f"❌ An error occurred during RNABERT processing setup or model loading: {e}")

except Exception as e: # Catching errors related to PyTorch import/device check
    print("❌ PyTorch failed to import or device check failed. Skipping RNABERT embeddings.")
    print("   Error:", e)


# If embeddings dict is non-empty, merge into your node features
if embeddings:
    print(f"⦿ Merging {len(embeddings)} embeddings into node features…")
    emb_df = pd.DataFrame.from_dict(
        embeddings, orient="index"
    )
    # Dynamically create column names for embeddings
    emb_cols = [f"rnabert_{i}" for i in range(emb_df.shape[1])]
    emb_df.columns = emb_cols
    emb_df.index.name = "rnacentral_id"

    try:
        features = pd.read_csv("data/node_augmented_features.csv",
                               index_col="rnacentral_id")

        # Ensure index types match - important for merging!
        features.index = features.index.astype(str)
        emb_df.index = emb_df.index.astype(str)

        combined = features.join(emb_df, how="left")

        # Fill NaNs specifically in the embedding columns with 0
        # Avoids filling NaNs that might exist in the original features
        combined[emb_cols] = combined[emb_cols].fillna(0)

        out = "data/node_with_rnabert.csv"
        combined.to_csv(out)
        print(f"✔️ Saved node_with_rnabert.csv ({combined.shape[0]} rows × {combined.shape[1]} columns)")
    except FileNotFoundError:
        print(f"❌ Error: data/node_augmented_features.csv not found. Cannot merge embeddings.")
    except Exception as e:
        print(f"❌ An error occurred during merging/saving: {e}")

# Refined final messages based on execution flow
elif 'graph_ids' in locals() and graph_ids and fasta is None:
     print("⚠️ FASTA file loading failed or skipped. RNABERT embeddings could not be generated.")
elif 'graph_ids' in locals() and not graph_ids:
     print("⚠️ No graph nodes found or edges file missing. RNABERT embeddings could not be generated.")
elif 'graph_ids' in locals() and graph_ids and fasta is not None and processed_count == 0:
    print("ℹ️ No embeddings were generated despite finding graph nodes and FASTA. Check for errors during embedding generation batches.")
else:
    print("⚠️ RNABERT embeddings were not generated due to setup errors (PyTorch/multimolecule import). Original node features were not modified.")

In [ ]:
#GENERATING node_all_features FEATURES#
# 1) Load each DataFrame
aug = pd.read_csv("data/node_augmented_features.csv", index_col="rnacentral_id") #mapped back partially on progress
km  = pd.read_csv("data/node_with_hashkmers.csv",       index_col="rnacentral_id") #mapped back 
rb  = pd.read_csv("data/node_with_rnabert.csv",         index_col="rnacentral_id")#mapped back

# 2) Identify embedding columns only (exclude any aug columns)
aug_cols = set(aug.columns)
km_emb   = [c for c in km.columns if c not in aug_cols]
rb_emb   = [c for c in rb.columns if c not in aug_cols]

# 3) Subset km & rb to only their unique embedding columns
km_only = km[km_emb]
rb_only = rb[rb_emb]

# 4) Now join all three on the index without collisions
merged = aug.join(km_only, how="inner") \
            .join(rb_only, how="inner") \
            .fillna(0)

print(f"Merged shape: {merged.shape[0]} nodes × {merged.shape[1]} features")
          
# 5) Save
merged.to_csv("data/node_all_features.csv")
print("✔️ Saved → data/node_all_features.csv")


## Edge Creation

In [ ]:
# ─── 0. Paths ───────────────────────────────────────────────────────────────
base_dir      = "/kaggle/input/gnn-edge-creation"
mapping_tsv   = os.path.join(base_dir, "id_mapping.tsv")
npinter_txt   = os.path.join(base_dir, "interaction_NPInterv5.txt")
node_csv       = os.path.join(base_dir, "node_gene_features.csv")
psicquic_file = os.path.join(base_dir, "psicquic.tsv")
gtf_filepath = os.path.join(base_dir, "gencode.v43.annotation.gtf")
ensembl_to_rnacentral = os.path.join(base_dir, "ensembl_to_rnacentral.csv")
out_edges_csv = "/kaggle/working/graph_edges_main.csv"
# Cell 1: Imports & file‐paths
# —– adjust these paths to wherever you’ve placed your files —–
GTF_FILE            = gtf_filepath
ENSEMBL_MAP_FILE    = ensembl_to_rnacentral             # unsure: delimiter?
NPINTER_FILE        = npinter_txt             # unsure: exact name/sep
OUTPUT_EDGE_FILE    = 'rnacentral_edges.csv'


In [ ]:
# Cell 2: Parse GTF → build transcript2gene (ENST → ENSG)
transcript2gene = {}
with open(GTF_FILE, 'r') as fh:
    for line in fh:
        if line.startswith('#'):
            continue
        cols = line.strip().split('\t', 8)
        if cols[2] != 'transcript':
            continue
        attr_str = cols[8]
        # build attribute dict
        attrs = {}
        for piece in attr_str.split(';'):
            piece = piece.strip()
            if not piece: 
                continue
            key, val = piece.split(' ', 1)
            attrs[key] = val.strip('"')
        enst = attrs.get('transcript_id', '').split('.')[0]
        ensg = attrs.get('gene_id',      '').split('.')[0]
        if enst and ensg:
            transcript2gene[enst] = ensg

print(f"Built transcript2gene for {len(transcript2gene)} transcripts")
# show a few examples
for tx, gene in list(transcript2gene.items())[:10]:
    print(f"  {tx} → {gene}")


In [ ]:
# Cell 3: Load Ensembl→URS mapping → gene2urs dict
map_df = pd.read_csv(ENSEMBL_MAP_FILE, dtype=str)   # unsure: sep=','
# strip version
map_df['external_id'] = map_df['external_id'].str.split('.').str[0]
# group in case multiple URS per gene
gene2urs = map_df.groupby('external_id')['rnacentral_id'].apply(list).to_dict()

print(f"Loaded mapping for {len(gene2urs)} ENSG genes")
# sample
{g: gene2urs[g] for g in list(gene2urs)[:5]}


In [ ]:
# Cell 4: Read NPInter, filter to human RNA–RNA
edges = pd.read_csv(NPINTER_FILE, sep='\t', dtype=str)   # unsure: sep
# filter human
edges = edges[edges['organism'].str.contains('9606|Homo sapiens', na=False)]
# filter RNA–RNA
edges = edges[edges['level'] == 'RNA-RNA']
print(f"Remaining NPInter RNA–RNA rows: {len(edges)}")
edges.head()


In [ ]:
# Cell 5b: build lookup dicts from id_mapping.tsv, including GeneCards & HGNC
# IDMAP_FILE = 'path/to/id_mapping.tsv'   # ← adjust path
CHUNKSIZE  = 200_000
IDMAP_FILE=mapping_tsv
# id_mapping.tsv cols: URS, database, external_id, taxid, rna_type
# colnames = ["rnacentral_id","database","external_id","taxid","rna_type"]
colnames = ["rnacentral_id","database","external_id","other_acc","taxid","rna_type"]
reader = pd.read_csv(
    IDMAP_FILE,
    sep="\t",
    header=None,
    names=colnames,
    usecols=["database","external_id","rnacentral_id"],
    dtype=str,
    chunksize=CHUNKSIZE,
)




In [ ]:
# Cell 5b: Build lookup dictionaries (NONCODE, RefSeq, miRBase, GeneCards, HGNC, ENSG)
IDMAP_FILE    = mapping_tsv            # ← adjust path
ENSEMBL_MAP   = ensembl_to_rnacentral # ← adjust path
CHUNKSIZE     = 200_000


# Initialize dicts
noncode2URS   = {}
refseq2URS    = {}
mirbase2URS   = {}
genecards2URS = {}
hgnc2URS      = {}

for chunk in reader:
    # normalize external_id
    chunk["external_id"] = (
        chunk["external_id"]
             .str.upper()
             .str.split(".")
             .str[0]
    )
    db = chunk["database"].str.upper()

    # NONCODE
    sub = chunk[db == "NONCODE"]
    for ext, grp in sub.groupby("external_id"):
        noncode2URS.setdefault(ext, set()).update(grp["rnacentral_id"])

    # RefSeq
    sub = chunk[db == "REFSEQ"]
    for ext, grp in sub.groupby("external_id"):
        refseq2URS.setdefault(ext, set()).update(grp["rnacentral_id"])

    # miRBase
    sub = chunk[db.str.contains("MIRBASE", na=False)]
    for ext, grp in sub.groupby("external_id"):
        mirbase2URS.setdefault(ext, set()).update(grp["rnacentral_id"])

    # GeneCards
    sub = chunk[db == "GENECARDS"]
    for ext, grp in sub.groupby("external_id"):
        genecards2URS.setdefault(ext, set()).update(grp["rnacentral_id"])

    # HGNC
    sub = chunk[db == "HGNC"]
    for ext, grp in sub.groupby("external_id"):
        hgnc2URS.setdefault(ext, set()).update(grp["rnacentral_id"])

# Load ENSG→URS
map_df = pd.read_csv(ENSEMBL_MAP, dtype=str)
map_df["external_id"] = map_df["external_id"].str.split(".").str[0]
gene2URS = map_df.groupby("external_id")["rnacentral_id"].apply(set).to_dict()

# Sanity prints
print(f"Built maps:")
print(f"  NONCODE → URS:   {len(noncode2URS)} entries")
print(f"  RefSeq → URS:    {len(refseq2URS)} entries")
print(f"  miRBase → URS:   {len(mirbase2URS)} entries")
print(f"  GeneCards → URS: {len(genecards2URS)} entries")
print(f"  HGNC → URS:      {len(hgnc2URS)} entries")
print(f"  ENSG → URS:      {len(gene2URS)} entries")

# Conditional check
if len(noncode2URS) < 200_000:
    print("⚠️  NONCODE map seems small—expected ~236k entries.")
if len(refseq2URS) < 90_000:
    print("⚠️  RefSeq map underpopulated.")
if len(mirbase2URS) < 80_000:
    print("⚠️  miRBase map underpopulated.")


This is the expected output

Built maps:
  NONCODE → URS:   236163 entries
  RefSeq → URS:    98310 entries
  miRBase → URS:   87467 entries
  GeneCards → URS: 691750 entries
  HGNC → URS:      8841 entries
  ENSG → URS:      56419 entries

In [ ]:
t2u = {}
for chunk in reader:
    chunk["external_id"] = chunk["external_id"].str.upper().str.split(".").str[0]
    sub = chunk[chunk["database"].str.upper() == "NONCODE"]
    for ext, grp in sub.groupby("external_id"):
        if ext.startswith("NONHSAT"):
            t2u.setdefault(ext, set()).update(grp["rnacentral_id"])

# build gene-level map: NONHSAG#### → union of NONHSAT#### URS
g2u = {}
for t_id, urs_set in t2u.items():
    # change transcript->gene: NONHSAT123456 -> NONHSAG123456
    g_id = "NONHSAG" + t_id[len("NONHSAT"):]
    g2u.setdefault(g_id, set()).update(urs_set)

print(f"Transcript-level entries: {len(t2u)}")
print(f"Gene-level entries:       {len(g2u)}")

In [ ]:
# Cell 5c: Final source+target mapping with sanity checks

# Map NPInter source NC codes (NONHSAG…) via g2u
def map_source(ncID):
    if not isinstance(ncID, str):
        return []
    key = ncID.strip().upper().split('.')[0]    # e.g. "NONHSAG040596"
    # lookup in gene‐level NONCODE→URS map (g2u)
    return list(g2u.get(key, []))

# Map target IDs as before
def map_target(tarID):
    if not isinstance(tarID, str):
        return []
    t = tarID.strip().upper().split('.')[0]
    if t in mirbase2URS:
        return list(mirbase2URS[t])
    if t in refseq2URS:
        return list(refseq2URS[t])
    if t.startswith("ENSG"):
        return list(gene2URS.get(t, []))
    return []

# Apply mappings
edges["source_URS_list"] = edges["ncID"].map(map_source)
edges["target_URS_list"] = edges["tarID"].map(map_target)

# Summarize
total    = len(edges)
src_cnt  = edges["source_URS_list"].map(len).gt(0).sum()
tgt_cnt  = edges["target_URS_list"].map(len).gt(0).sum()

print(f"Sources mapped: {src_cnt} / {total} ({src_cnt/total:.2%})")
print(f"Targets mapped: {tgt_cnt} / {total} ({tgt_cnt/total:.2%})")

# Conditional warning if off-nominal
if not (45_000 <= src_cnt <= 55_000):
    print("⚠️ WARNING: Expected ~49k sources mapped—please recheck your g2u dict.")
if not (120_000 <= tgt_cnt <= 130_000):
    print("⚠️ WARNING: Expected ~122k targets mapped—please recheck RefSeq/miRBase mapping.")


In [ ]:
# Cell 5d: explode lists and write edge list
# ------------------------------------------------
# Assumes edges DataFrame has source_URS_list & target_URS_list

# 1) Explode multi‐valued lists into one row per URS–URS pair
df_edges = (
    edges
    .explode("source_URS_list")
    .explode("target_URS_list")
    .rename(columns={
        "source_URS_list": "source_URS",
        "target_URS_list": "target_URS"
    })
)

# 2) Drop rows where either side failed to map
df_edges = df_edges.dropna(subset=["source_URS","target_URS"])

# 3) (Optional) Remove self‐loops
df_edges = df_edges[df_edges.source_URS != df_edges.target_URS]

# 4) Drop duplicates
df_edges = df_edges[["source_URS","target_URS"]].drop_duplicates()

print("Final URS–URS edges:", len(df_edges))

# 5) Save to CSV
OUT_FILE = "rnacentral_edges.csv"
df_edges.to_csv(OUT_FILE, index=False)
print(f"Saved edges to {OUT_FILE}")


In [ ]:
# Cell 5d.1: Sanity‐check node/edge compatibility before building the graph
# Paths – adjust if needed
NODE_FILE = "/kaggle/input/gnn-edge-creation/node_gene_features.csv"
EDGE_FILE = "/kaggle/working/rnacentral_edges.csv"

# 1) Load nodes
nodes_df = pd.read_csv(NODE_FILE, usecols=["rnacentral_id"])
node_set = set(nodes_df["rnacentral_id"])
print(f"Total nodes available: {len(node_set)}")

# 2) Load edges
edges_df = pd.read_csv(EDGE_FILE)
src_ids = set(edges_df["source_URS"])
tgt_ids = set(edges_df["target_URS"])
print(f"Total edges: {len(edges_df)}")
print(f"Unique source URS in edges: {len(src_ids)}")
print(f"Unique target URS in edges: {len(tgt_ids)}")

# 3) Find any orphan nodes in edges
missing_src = src_ids - node_set
missing_tgt = tgt_ids - node_set
print(f"Orphan source URS (in edges but not in nodes): {len(missing_src)}")
print(f"Orphan target URS (in edges but not in nodes): {len(missing_tgt)}")
if missing_src:
    print("  Examples:", list(missing_src)[:10])
if missing_tgt:
    print("  Examples:", list(missing_tgt)[:10])

# 4) (Optional) prune edges to only those with both ends in node_set
pruned_edges = edges_df[
    edges_df["source_URS"].isin(node_set) &
    edges_df["target_URS"].isin(node_set)
]
print(f"Edges after pruning orphans: {len(pruned_edges)} "
      f"(dropped {len(edges_df)-len(pruned_edges)})")

# You can then feed `pruned_edges` into your graph builder:
# edge_index = torch.tensor([
#     pruned_edges.source_URS.map(id2idx).values,
#     pruned_edges.target_URS.map(id2idx).values
# ], dtype=torch.long)


## Graph and Model Creation

In [ ]:
import torch
import pandas as pd
import numpy as np
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, VGAE, SAGEConv
from torch_geometric.utils import train_test_split_edges
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm
import os
from collections import defaultdict
import gzip
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.decomposition import IncrementalPCA
from scipy import sparse
import re
from functools import reduce
# -------------------------------------------------------------
# SECTION 1: Setup and Imports
# -------------------------------------------------------------


print("--- Section 1: Setup and Imports ---")

# Set device (GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Hyperparameters
OUT_DIM = 64      # Latent space dimension
LEARNING_RATE = 0.01
NUM_EPOCHS = 100
VAL_RATIO = 0.1   # Ratio for validation edges
TEST_RATIO = 0.1  # Ratio for test edges
TOP_K_NOVEL = 1000 # Number of top novel links to predict
SCORE_BATCH_SIZE = 10000 # Batch size for scoring all pairs

# File paths (Update these based on your Kaggle data paths)
NODE_FEATURES_PATH = "/kaggle/input/graph-blocks/node_all_features.csv" # Assuming this has rnacentral_id + numeric features
EDGES_PATH = "/kaggle/input/graph-blocks/rnacentral_edges.csv"
MODEL_SAVE_PATH = "vgae_rnabert_model.pt" # Output path for the saved model

print("Setup complete. Imports loaded.")


# -------------------------------------------------------------
# SECTION 2: Data Loading and Preparation
# -------------------------------------------------------------
print("\n--- Section 2: Data Loading and Preparation ---")

try:
    # 2.1) Load merged node features (assuming RNABERT embeddings are already included or separate features are ready)
    # This DataFrame should have 'rnacentral_id' and numeric feature columns
    df_features = pd.read_csv(NODE_FEATURES_PATH)
    print(f"Loaded node features: {df_features.shape[0]} nodes, {df_features.shape[1]} columns.")

    # Store node IDs and create ID to index map
    node_ids = df_features["rnacentral_id"].tolist()
    id2idx = {nid: i for i, nid in enumerate(node_ids)}

    # Select only numeric feature columns and convert to tensor
    # Ensure 'rnacentral_id' is excluded
    numeric_cols = df_features.select_dtypes(include=np.number).columns.tolist()
    if 'rnacentral_id' in numeric_cols:
         numeric_cols.remove('rnacentral_id')

    x = torch.tensor(df_features[numeric_cols].values, dtype=torch.float)
    print(f"Node features tensor (x): {x.shape}")

    # 2.2) Load edges
    df_edges = pd.read_csv(EDGES_PATH)
    print(f"Loaded edges: {df_edges.shape[0]} edges.")

    # Convert node IDs in edges to indices using the id2idx map
    # Ensure all edge IDs are present in node_ids, handle missing if necessary
    try:
        src = [id2idx[s] for s in df_edges.source_URS]
        dst = [id2idx[t] for t in df_edges.target_URS]
        edge_index = torch.tensor([src, dst], dtype=torch.long)
        print(f"Edge index tensor: {edge_index.shape}")
    except KeyError as e:
        print(f"❌ Error: Edge ID not found in node features: {e}. Check data consistency.")
        # Handle this error appropriately, maybe exit or filter edges
        exit() # Exit for simplicity here


    # 2.3) Create the full PyTorch Geometric Data object
    # This data object represents the graph BEFORE any train/test split
    data_full = Data(x=x, edge_index=edge_index)
    print("Full graph Data object created:", data_full)

    # 2.4) Perform train/validation/test edge split
    # train_test_split_edges operates on CPU, so we'll work with a CPU copy
    data_cpu = data_full.clone().cpu()
    data_lp = train_test_split_edges(data_cpu, val_ratio=VAL_RATIO, test_ratio=TEST_RATIO)
    print("\nEdge split performed.")
    print(f"Train positive edges: {data_lp.train_pos_edge_index.size(1)}")
    print(f"Validation positive edges: {data_lp.val_pos_edge_index.size(1)}")
    print(f"Validation negative edges: {data_lp.val_neg_edge_index.size(1)}")
    print(f"Test positive edges: {data_lp.test_pos_edge_index.size(1)}")
    print(f"Test negative edges: {data_lp.test_neg_edge_index.size(1)}")


except FileNotFoundError as e:
    print(f"❌ Data file not found: {e}. Please check paths.")
    exit() # Exit if data files are missing
except Exception as e:
    print(f"❌ An error occurred during data loading or preparation: {e}")
    exit()

print("Data loading and preparation complete.")


# -------------------------------------------------------------
# SECTION 3: Model Definition
# -------------------------------------------------------------
print("\n--- Section 3: Model Definition ---")

class Encoder(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1     = GCNConv(in_channels, 2*out_channels)
        self.conv_mu   = GCNConv(2*out_channels, out_channels)
        self.conv_logv = GCNConv(2*out_channels, out_channels)

    def forward(self, x, edge_index):
        x = torch.relu(self.conv1(x, edge_index))
        return (
            self.conv_mu(x, edge_index),
            self.conv_logv(x, edge_index)
        )

print("Encoder model class defined.")


# -------------------------------------------------------------
# SECTION 4: Model Instantiation and Training
# -------------------------------------------------------------
print("\n--- Section 4: Model Instantiation and Training ---")

# Instantiate model
# Use data_full.num_features as the input dimension
model = VGAE(Encoder(data_full.num_features, OUT_DIM)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Instantiated VGAE model with {data_full.num_features} input features and {OUT_DIM} latent dimensions.")
print(f"Model is on device: {next(model.parameters()).device}")
print(f"Starting training for {NUM_EPOCHS} epochs...")
loss_history = []
val_auc_history = []
val_ap_history  = []
# We’ll collect labels/scores once for final ROC/PR
final_y_true = []
final_y_score = []

val_y_true, val_y_score = [], []
# Training Loop
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    # Move data required for the training step to the device
    # data_lp.x is node features for all nodes, train_pos_edge_index are the edges to train on
    x_on_device = data_lp.x.to(device)
    train_pos_edge_index_on_device = data_lp.train_pos_edge_index.to(device)

    # Encode using node features and training edges
    z = model.encode(x_on_device, train_pos_edge_index_on_device)

    # Calculate loss (reconstruction loss on training edges + KL divergence)
    loss = (
        model.recon_loss(z, train_pos_edge_index_on_device) + # Reconstruction loss on positive training edges
        (1 / data_lp.num_nodes) * model.kl_loss()           # KL divergence loss
    )

    # Backpropagation and optimization
    loss.backward()
    optimizer.step()

    # 4.1) Evaluation (Periodic)
    # Evaluate on validation and test sets every 10 epochs or on the first epoch
    # if epoch % 10 == 0 or epoch == 1:
    model.eval()
    with torch.no_grad():
        # Encode again using the training edges to get current embeddings for evaluation
        z_eval = model.encode(data_lp.x.to(device), data_lp.train_pos_edge_index.to(device))

        # Helper function to score edge predictions
        def score_edges(edges_tensor, embeddings):
             # Move evaluation edges to the device, compute scores, move back to CPU, convert to numpy
             return model.decoder(embeddings, edges_tensor.to(device)).sigmoid().cpu().numpy()

        # Validation Evaluation
        pos_val = data_lp.val_pos_edge_index
        neg_val = data_lp.val_neg_edge_index
        y_val = [1]*pos_val.size(1) + [0]*neg_val.size(1)
        # Score positive and negative validation edges
        preds_val = list(score_edges(pos_val, z_eval)) + list(score_edges(neg_val, z_eval))
        val_y_true.extend(y_val)
        val_y_score.extend(preds_val)
        val_auc = roc_auc_score(y_val, preds_val)
        val_ap = average_precision_score(y_val, preds_val)
        val_auc_history.append(val_auc)
        val_ap_history.append(val_ap)

        # Test Evaluation (generally do this only ONCE at the end)
        # Including here as it was in your snippet, but be cautious.
        pos_test = data_lp.test_pos_edge_index
        neg_test = data_lp.test_neg_edge_index
        y_test = [1]*pos_test.size(1) + [0]*neg_test.size(1)
        # Score positive and negative test edges
        preds_test = list(score_edges(pos_test, z_eval)) + list(score_edges(neg_test, z_eval))
        test_auc = roc_auc_score(y_test, preds_test)
        test_ap = average_precision_score(y_test, preds_test)
        if epoch == NUM_EPOCHS or (epoch % 10 == 0 and epoch + 10 > NUM_EPOCHS):
            final_y_true = y_val
            final_y_score = preds_val

        # Print progress and metrics
        print(f"Epoch {epoch:03d} — Loss: {loss.item():.4f} | Val ROC–AUC: {val_auc:.4f}, AP: {val_ap:.4f} | Test ROC–AUC: {test_auc:.4f}, AP: {test_ap:.4f}")
    # else:
    #      # Print training loss for other epochs without full evaluation
    #      print(f"Epoch {epoch:03d} — Loss: {loss.item():.4f}")
    loss_history.append(loss.item())


print("\nTraining complete.")


# -------------------------------------------------------------
# SECTION 5: Save the Trained Model
# -------------------------------------------------------------
print("\n--- Section 5: Saving the Trained Model ---")

try:
    # Save only the model's state dictionary (recommended)
    torch.save(model.state_dict(), MODEL_SAVE_PATH)
    print(f"✔️ Saved model state dictionary to {MODEL_SAVE_PATH}")
except Exception as e:
    print(f"❌ Error saving model: {e}")

print("Model saving complete.")


# -------------------------------------------------------------
# SECTION 6: Inference - Finding Top K Novel Predictions
# -------------------------------------------------------------
print("\n--- Section 6: Finding Top K Novel Predictions ---")

# Ensure model is in evaluation mode
model.eval()
with torch.no_grad():
    # Re-encode using the full graph's edges to get embeddings for all nodes
    # This representation is used for predicting ALL possible links
    # Ensure data_full has been moved to the correct 'device'
    # If data_full was large, it might be better to keep it on CPU initially and move parts as needed
    # But for encoding the full graph, you usually need the whole thing on the device if possible.
    # Assuming data_full is already on the device from Section 2.4 or move it here:
    data_full = data_full.to(device) # Ensure data_full is on device
    z_final = model.encode(data_full.x, data_full.edge_index)
    print(f"Final embeddings (z) for all {data_full.num_nodes} nodes computed.")

    # 6.1) Generate all unique pairs (upper triangle)
    N = data_full.num_nodes
    print(f"⦿ Generating all {N*(N-1)//2} unique pairs for scoring...")
    # Using list comprehension to generate pairs on CPU
    all_pairs_list = [(i, j) for i in range(N) for j in range(i+1, N)]
    print(f"✔️ Generated {len(all_pairs_list)} pairs.")

    # 6.2) Calculate scores for all pairs in batches
    print(f"⦿ Calculating scores for all pairs in batches (Batch size: {SCORE_BATCH_SIZE})...")
    scores_all_list = []
    # Use tqdm to monitor batch processing for scoring
    for i in tqdm(range(0, len(all_pairs_list), SCORE_BATCH_SIZE), desc="Scoring batches"):
        batch_pairs = all_pairs_list[i : i + SCORE_BATCH_SIZE]
        # Convert batch of pairs to tensor, transpose (shape [2, batch_size]), and move to device
        batch_pairs_tensor = torch.tensor(batch_pairs).t().to(device)

        # Calculate scores for the batch using the decoder
        batch_scores = model.decoder(z_final, batch_pairs_tensor).sigmoid().cpu().numpy() # Scores back to CPU/NumPy

        scores_all_list.append(batch_scores)

    # Concatenate scores from all batches into a single NumPy array
    scores_all = np.concatenate(scores_all_list)
    print("✔️ Finished scoring all pairs.")

    # 6.3) Combine pairs and scores and sort
    print("⦿ Combining pairs and scores and sorting...")
    # Create a list of (score, pair) tuples
    scored_pairs = [(scores_all[i], all_pairs_list[i]) for i in range(len(all_pairs_list))]

    # Sort by score in descending order
    scored_pairs.sort(key=lambda item: item[0], reverse=True)
    print("✔️ Finished sorting predictions by score.")

    # 6.4) Prepare original edges for fast lookup
    print("⦿ Preparing original edges for novelty check...")
    # Need the original edge_index from the full graph, on CPU
    # Ensure data_full.edge_index is on CPU for conversion to set
    original_edges_tensor_cpu = data_full.edge_index.cpu()

    # Create a set of tuples for quick lookup (canonical form: smaller, larger index)
    original_edge_set = set()
    for i in range(original_edges_tensor_cpu.size(1)):
        u, v = original_edges_tensor_cpu[0, i].item(), original_edges_tensor_cpu[1, i].item()
        original_edge_set.add((min(u, v), max(u, v)))

    print(f"✔️ Prepared set of {len(original_edge_set)} original edges for lookup.")


    # 6.5) Select the top K novel predictions
    print(f"⦿ Selecting top {TOP_K_NOVEL} novel predictions...")
    top_k_novel_predictions = []
    # Iterate through the sorted predictions from highest score downwards
    # Use tqdm to show progress through the sorted list
    for score, pair in tqdm(scored_pairs, desc=f"Finding top {TOP_K_NOVEL} novel"):
        u, v = pair
        # Get the canonical form of the predicted pair (u, v)
        canonical_pair = (min(u, v), max(u, v))

        # Check if this canonical pair is NOT in the set of original edges
        if canonical_pair not in original_edge_set:
            # It's a novel prediction! Add it to our list
            top_k_novel_predictions.append((u, v, score)) # Store the pair (in its original form) and its score

            # Check if we have found K novel predictions
            if len(top_k_novel_predictions) == TOP_K_NOVEL:
                break # Stop searching once K is found

    print(f"✔️ Found {len(top_k_novel_predictions)} top novel predictions.")

# --- 6.6) Output the top K novel predictions ---
print("\n--- Top K Novel ncRNA–ncRNA Predictions ---")
if top_k_novel_predictions:
    # Convert to a pandas DataFrame
    top_k_df = pd.DataFrame(top_k_novel_predictions, columns=["Source_Node_Index", "Target_Node_Index", "Prediction_Score"])

    print("Top K Novel Predictions (DataFrame Head):")
    print(top_k_df.head())

    # You can save this DataFrame to a CSV output file
    # top_k_output_path = "top_k_novel_predictions.csv"
    # try:
    #     top_k_df.to_csv(top_k_output_path, index=False)
    #     print(f"\nSaved top novel predictions to {top_k_output_path}")
    # except Exception as e:
    #      print(f"❌ Error saving top predictions: {e}")

else:
    print("No novel predictions found (or TOP_K_NOVEL was 0).")

print("\nInference complete.")

# -------------------------------------------------------------
# SECTION 7: Code for Loading a Saved Model (Example for future use)
# -------------------------------------------------------------
print("\n--- Section 7: Example Code for Loading a Saved Model ---")
print("This section is an example and is NOT executed as part of the main run.")
print("You would use this in a *separate* script or notebook cell to load the .pt file saved in Section 5.")

"""
# Example of loading the model state dictionary:
# --- Requires ---
# import torch
# from torch_geometric.nn import VGAE, GCNConv # Need model class definitions
# from torch_geometric.data import Data # If you need to load data for inference
# Assuming Encoder class is also defined or imported

# --- Configuration ---
# You MUST know the input and output dimensions used during training
# in_channels = ??? (e.g., data_full.num_features used during training)
# out_dim = 64 (matching OUT_DIM used during training)
# model_save_path = "vgae_rnabert_model.pt"

# --- Set Device (where you want the loaded model to run) ---
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Instantiate a new model with the SAME architecture ---
# loaded_model = VGAE(Encoder(in_channels, out_dim))

# --- Load the saved state dictionary ---
# try:
#     state_dict = torch.load(model_save_path, map_location=device)
#     loaded_model.load_state_dict(state_dict)
#     loaded_model.to(device) # Ensure model is on the correct device
#     loaded_model.eval()     # Set to evaluation mode

#     print(f"Successfully loaded model from {model_save_path}")

#     # Now you can use loaded_model for inference
#     # For example, load data, get node embeddings:
#     # loaded_data_full = ... load your data here ...
#     # loaded_data_full = loaded_data_full.to(device)
#     # z_loaded = loaded_model.encode(loaded_data_full.x, loaded_data_full.edge_index)
#     # ... then use z_loaded for predictions ...

# except FileNotFoundError:
#     print(f"Error loading model: {model_save_path} not found.")
# except Exception as e:
#     print(f"Error loading model: {e}")
"""
print("Example loading code complete (not executed).")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import umap
from torch_geometric.utils import to_networkx
from networkx.algorithms.community import louvain_communities

# --- 1) Load or prepare data ---
# z_final: torch.Tensor of shape [num_nodes, latent_dim]
# data_full: your PyG Data object containing edge_index
# node_ids: list of rnacentral IDs in the same order as z_final
# mndr_df: DataFrame with columns ["rnacentral_id","schizo_associated"] (bool or 0/1)

# Example (uncomment & adjust paths if needed):
# import torch
# z_final = torch.load("data/z_final.pt")             # embeddings tensor
df_feat = pd.read_csv("/kaggle/input/graph-blocks/node_all_features.csv")
node_ids = df_feat["rnacentral_id"].tolist()
mndr_features = pd.read_csv("/kaggle/input/graph-blocks/node_mndr_features.csv", dtype={"rnacentral_id": str})

# 2) Create a binary “schizo_associated” column 
#    (e.g. any nonzero MNDR score → True, else False)
mndr_features["schizo_associated"] = mndr_features["mndr_score"].gt(0)

# 3) Keep only the two needed columns
mndr_df = mndr_features[["rnacentral_id", "schizo_associated"]]

# 4) (Optional) Verify
print(f"Total nodes with MNDR data: {len(mndr_df)}")
print("Sample:")
print(mndr_df.head())

# --- 2) Build embedding DataFrame ---
# Build DataFrame of embeddings + MNDR labels
emb_df = pd.DataFrame(
    z_final.cpu().numpy(),
    index=node_ids,
    columns=[f"z{i}" for i in range(z_final.size(1))]
)

# Now left-join schizo labels; unmatched → NaN → fill False
emb_df = emb_df.merge(
    mndr_df.set_index("rnacentral_id"),
    left_index=True, right_index=True,
    how="left"
)
emb_df["schizo_associated"] = emb_df["schizo_associated"].fillna(False)


# --- 2) Compute Louvain communities with NetworkX ---
G = to_networkx(data_full, to_undirected=True)
# returns a list of sets of node indices
communities = louvain_communities(G, weight='weight', resolution=1.0)

# build a mapping node_idx → community_id
partition = {}
for comm_id, comm_nodes in enumerate(communities):
    for idx in comm_nodes:
        partition[idx] = comm_id

# map back to URS IDs
idx2id = {i: nid for i, nid in enumerate(node_ids)}
emb_df["louvain_comm"] = [
    partition[i] for i in range(len(node_ids))
]

# --- 3) UMAP projection ---
reducer = umap.UMAP(n_components=2, random_state=42)
coords = reducer.fit_transform(emb_df[[f"z{i}" for i in range(z_final.size(1))]])
emb_df["UMAP1"], emb_df["UMAP2"] = coords[:,0], coords[:,1]

# --- 4) Plotting ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

# Left: schizophrenia‐associated (red/gray)
colors = emb_df["schizo_associated"].map({True:"red", False:"gray"})
axes[0].scatter(emb_df["UMAP1"], emb_df["UMAP2"], c=colors, s=10, alpha=0.7)
axes[0].set_title("MNDR‐Annotated Schizophrenia ncRNAs")
axes[0].grid(True, linestyle=":", alpha=0.5)
axes[0].set_xlabel("UMAP1")
axes[0].set_ylabel("UMAP2")
# Legend
red_patch = mpatches.Patch(color='red', label='Schizo-associated')
gray_patch = mpatches.Patch(color='lightgray', label='Other ncRNAs')
axes[0].legend(handles=[red_patch, gray_patch], loc="upper right")
# Right: Louvain communities
n_comms = emb_df["louvain_comm"].nunique()
cmap   = plt.get_cmap("tab20", n_comms)
axes[1].scatter(
    emb_df["UMAP1"], emb_df["UMAP2"],
    c=emb_df["louvain_comm"], cmap=cmap,
    s=10, alpha=0.7
)
axes[1].set_title("Louvain Community Assignments")
axes[1].set_xlabel("UMAP1")
axes[1].set_ylabel("UMAP2")
axes[1].grid(True, linestyle=":", alpha=0.5)
# Colorbar for communities
norm = mcolors.Normalize(vmin=0, vmax=n_comms-1)
cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap),
                    ax=axes[1], fraction=0.046, pad=0.04)
cbar.set_label("Community ID")
plt.tight_layout()
plt.savefig("/kaggle/working/latent_umap.pdf", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, precision_recall_curve, auc

# 3.1) Compute ROC & PR on the held-out validation set
fpr, tpr, _   = roc_curve(final_y_true, final_y_score)
prec, rec, _  = precision_recall_curve(final_y_true, final_y_score)
roc_auc_val   = auc(fpr, tpr)
pr_auc_val    = auc(rec, prec)

# 3.2) Prepare epoch axis for history
epochs = np.arange(1, len(loss_history) + 1)

# 3.3) Create 2×2 figure, leaving [1,1] blank
fig, axes = plt.subplots(2, 2, figsize=(12,10))
ax_roc, ax_pr, ax_loss = axes[0,0], axes[0,1], axes[1,0]
axes[1,1].axis('off')

# (a) ROC Curve
ax_roc.plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc_val:.3f}")
ax_roc.plot([0,1], [0,1], '--', color='gray')
ax_roc.set_title("ROC Curve")
ax_roc.set_xlabel("False Positive Rate")
ax_roc.set_ylabel("True Positive Rate")
ax_roc.legend(loc="lower right")

# (b) Precision–Recall Curve
ax_pr.plot(rec, prec, lw=2, label=f"AP = {pr_auc_val:.3f}")
ax_pr.set_title("Precision–Recall Curve")
ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.legend(loc="upper right")

# (c) Loss & Validation AUC over Epochs
ax_loss.plot(epochs, loss_history, label="Train Loss", alpha=0.7)
ax_loss.plot(epochs, val_auc_history, label="Val ROC–AUC", color='red')
ax_loss.set_title("Training Loss & Validation AUC")
ax_loss.set_xlabel("Epoch")
ax_loss.legend(loc="best")
# ax2 = ax_loss.twinx()
# ax2.plot(eval_epochs, val_auc_history, color='red', label='Val AUC')
# ax2.set_ylabel('Val ROC–AUC')

plt.tight_layout()
plt.savefig("/kaggle/working//roc_pr_training.png", dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, precision_recall_curve, auc


plt.figure(figsize=(7, 6)) # Create a new figure for this plot
plt.plot(fpr, tpr, color='blue', lw=3, label=f"AUC = {roc_auc_val:.3f}")
plt.plot([0,1],[0,1], '--', color='gray')
plt.xlabel("False Positive Rate") # Correct way for plt
plt.ylabel("True Positive Rate")  # Correct way for plt
plt.title("(a) ROC Curve")       # Correct way for plt
plt.legend(loc="lower right")
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig("/kaggle/working/roc_curve.pdf", dpi=300)
plt.show()

print(f"ROC AUC on final validation set: {roc_auc_val:.3f}")
print(f"PR AUC on final validation set: {pr_auc_val:.3f}")

In [ ]:
# 4b) Precision–Recall Curve (Corrected for standalone plot)
plt.figure(figsize=(7, 6)) # Create a new figure for this plot

# Filter out rec=0 spike (as per your original code)
mask = rec > 0
plt.plot(rec[mask], prec[mask], color='green', lw=3, label=f"AP = {pr_auc_val:.3f}")
plt.xlabel("Recall")      # Correct way for plt
plt.ylabel("Precision")   # Correct way for plt
plt.title("(b) Precision–Recall Curve") # Correct way for plt
plt.legend(loc="lower left")
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig("/kaggle/working/pr_curve.pdf", dpi=300)
plt.show()

print(f"PR AUC on final validation set: {pr_auc_val:.3f}")

In [ ]:
# 4c) Training Loss & Val ROC–AUC (Corrected for standalone plot)
fig, ax_loss = plt.subplots(figsize=(10, 6)) # Create the figure and the primary axes (ax_loss)

ax_loss.plot(epochs, loss_history, color='purple', lw=2, alpha=0.7, label="Train Loss")
ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("Loss")
ax_loss.set_yscale('log')  # log scale to compress spikes

# Create a second y-axis that shares the same x-axis
ax2 = ax_loss.twinx()
ax2.plot(epochs, val_auc_history, 'o-', color='red', lw=2, label="Val ROC–AUC")
ax2.set_ylabel("Val ROC–AUC")

ax_loss.set_title("(c) Training Loss (log) & Validation AUC")

# Combine legends from both axes
lines, labels = ax_loss.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax_loss.legend(lines + lines2, labels + labels2, loc="best")

ax_loss.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig("/kaggle/working/training_curves.pdf", dpi=300)
plt.show()

In [ ]:
# 4d) Validation AP over Epochs (Corrected for standalone plot)
fig, ax_ap_epoch = plt.subplots(figsize=(10, 6)) # Create the figure and the axes

ax_ap_epoch.plot(epochs, val_ap_history, 's-', color='orange', lw=2, label="Val AP")
ax_ap_epoch.set_xlabel("Epoch")
ax_ap_epoch.set_ylabel("Average Precision")
ax_ap_epoch.set_title("(d) Validation AP over Epochs")
ax_ap_epoch.legend(loc="lower right")
ax_ap_epoch.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig("/kaggle/working/ap_epoch.pdf", dpi=300)
plt.show()